# 超参数调优：找到最佳配置
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：12_模型评估与选择 | 源文件：超参数调优.py | 核心功能：GridSearchCV、RandomizedSearchCV、调参策略

## 概述

超参数不能通过训练数据自动学习，需要手动或自动搜索。GridSearchCV 穷举所有组合，RandomizedSearchCV 随机采样。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
# --- 导入库 ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# 生成数据
X, y = make_classification(
    n_samples=500, n_features=15, n_informative=8,
    n_classes=2, random_state=42
)

# 5折分层交叉验证
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 数学原理

### 1. 超参数调优的数学定义

超参数 $\lambda$（如学习率、树深度、正则化系数）不能从数据中学习，需要在训练前设定。目标：

$$\lambda^* = \arg\min_{\lambda \in \Lambda} \mathbb{E}_{D_{val}}[L(y, f_{\lambda}(x))]$$

通过交叉验证估计验证误差：

$$\hat{\lambda} = \arg\min_{\lambda \in \Lambda} \frac{1}{K}\sum_{k=1}^{K} L_{val}^{(k)}(\lambda)$$

### 2. 网格搜索（GridSearchCV）

穷举所有参数组合：

$$\Lambda = \Lambda_1 \times \Lambda_2 \times \cdots \times \Lambda_p$$

总评估次数 $|\Lambda| = \prod_i |\Lambda_i|$。

代码中 `param_grid` 定义了 $3 \times 4 \times 3 = 36$ 种组合，每种做 5 折 CV，共训练 $36 \times 5 = 180$ 个模型。

**优点**：保证找到 $\Lambda$ 中的最优解
**缺点**：指数级增长（维度灾难）

### 3. 随机搜索（RandomizedSearchCV）

从参数分布中随机采样 $n$ 次：

$$\lambda^{(j)} \sim p(\lambda), \quad j = 1, \ldots, n$$

**理论优势**（Bergstra & Bengio, 2012）：如果只有少数参数对性能有显著影响，随机搜索比网格搜索更高效，因为它对每个参数独立采样，不会因无关参数的增加而指数增长。

### 4. 参数分布类型

| 分布 | 代码 | 采样方式 |
|------|------|----------|
| 离散均匀 | `[10, 50, 100, 200]` | 均匀随机选择 |
| 连续均匀 | `uniform(0.01, 1.0)` | $\lambda \sim U(a, b)$ |
| 对数均匀 | `loguniform(1e-4, 1e-1)` | $\log\lambda \sim U(\log a, \log b)$ |

对数均匀适合学习率等在对数尺度上有意义的参数。

### 5. 早停策略（逐步淘汰）

`HalvingGridSearchCV` 用少量资源评估所有候选，逐步淘汰差的：

1. 用 $n/K$ 个样本训练所有候选
2. 保留 top $1/\eta$ 个候选
3. 给保留的候选更多资源，重复直到剩一个

### 6. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `GridSearchCV(estimator, param_grid, cv=5)` | 穷举 $\Lambda$，5 折 CV |
| `RandomizedSearchCV(estimator, param_dist, n_iter=20)` | 随机采样 20 个 $\lambda$ |
| `gs.best_params_` | $\hat{\lambda} = \arg\min_\lambda \text{CV}(\lambda)$ |
| `gs.best_score_` | $\min_\lambda \frac{1}{K}\sum_k L_{val}^{(k)}(\lambda)$ |
| `gs.best_estimator_` | 在 $\hat{\lambda}$ 下重新训练的模型 |
| `cv_results_["mean_test_score"]` | 所有参数组合的 CV 分数 |

### 1. GridSearchCV 网格搜索

运行 1. GridSearchCV 网格搜索 的代码，观察算法在该环节的行为。

In [ ]:
print("=" * 60)
print("1. GridSearchCV 网格搜索")
print("=" * 60)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
}

# 计算总组合数
total_combos = 1
for v in param_grid.values():
    total_combos *= len(v)
print(f"参数组合总数: {total_combos}")
print(f"参数空间: {param_grid}")
# --- 输出结果 ---
print()

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
# --- 赋值/配置 ---
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
)

grid_search.fit(X, y)

print(f"最佳参数: {grid_search.best_params_}")
print(f"最佳交叉验证分数: {grid_search.best_score_:.4f}")
print(f"评估的参数组合数: {len(grid_search.cv_results_['params'])}")

# 显示Top5结果
print("\nTop5参数组合:")
results = grid_search.cv_results_
sorted_idx = np.argsort(results['rank_test_score'])[:5]
for idx in sorted_idx:
    print(f"  排名{results['rank_test_score'][idx]}: "
# --- 赋值/配置 ---
          f"验证分数={results['mean_test_score'][idx]:.4f} "
          f"训练分数={results['mean_train_score'][idx]:.4f} "
          f"参数={results['params'][idx]}")

### 2. RandomizedSearchCV 随机搜索

运行 2. RandomizedSearchCV 随机搜索 的代码，观察算法在该环节的行为。

In [ ]:
from scipy.stats import randint, uniform

print("\n" + "=" * 60)
print("2. RandomizedSearchCV 随机搜索")
print("=" * 60)

param_distributions = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(3, 50),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
# --- 继续 ---
    'max_features': uniform(0.1, 0.9),
}

n_iter = 30  # 随机采样30次
print(f"采样次数: {n_iter}")
print(f"参数分布:")
for name, dist in param_distributions.items():
    print(f"  {name}: {type(dist).__name__}")
# --- 输出结果 ---
print()

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=cv,
# --- 赋值/配置 ---
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=0,
    return_train_score=True,
# --- 继续 ---
)

random_search.fit(X, y)

print(f"最佳参数: {random_search.best_params_}")
print(f"最佳交叉验证分数: {random_search.best_score_:.4f}")

print("\nTop5参数组合:")
results_r = random_search.cv_results_
sorted_idx_r = np.argsort(results_r['rank_test_score'])[:5]
for idx in sorted_idx_r:
    params_str = {k: v for k, v in results_r['params'][idx].items()
# --- 条件判断 ---
                  if isinstance(v, (int, float, str, type(None)))}
    print(f"  排名{results_r['rank_test_score'][idx]}: "
          f"验证分数={results_r['mean_test_score'][idx]:.4f} "
          f"参数={params_str}")

### 3. 网格搜索 vs 随机搜索对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n" + "=" * 60)
print("3. 网格搜索 vs 随机搜索对比")
print("=" * 60)

print(f"{'方法':<20} {'最佳分数':>10} {'最佳参数数':>10}")
print("-" * 45)
print(f"  {'网格搜索':<18} {grid_search.best_score_:>10.4f} {total_combos:>10}")
print(f"  {'随机搜索(30次)':<18} {random_search.best_score_:>10.4f} {n_iter:>10}")

### 4. 不同模型的超参数调优

探索关键超参数对模型行为的影响。

In [ ]:
print("\n" + "=" * 60)
print("4. 不同学习率的GBDT调优")
print("=" * 60)

from sklearn.ensemble import GradientBoostingClassifier

gbdt_param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5],
}

gbdt_grid = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=gbdt_param_grid,
    cv=cv,
    scoring='accuracy',
# --- 赋值/配置 ---
    n_jobs=-1,
    verbose=0,
)

gbdt_grid.fit(X, y)

print(f"GBDT最佳参数: {gbdt_grid.best_params_}")
print(f"GBDT最佳分数: {gbdt_grid.best_score_:.4f}")

print("\n所有组合结果:")
results_gb = gbdt_grid.cv_results_
sorted_idx_gb = np.argsort(results_gb['rank_test_score'])
for rank, idx in enumerate(sorted_idx_gb):
    marker = " <-- 最佳" if rank == 0 else ""
# --- 输出结果 ---
    print(f"  排名{results_gb['rank_test_score'][idx]:>2}: "
          f"分数={results_gb['mean_test_score'][idx]:.4f} "
          f"参数={results_gb['params'][idx]}{marker}")

## 关键代码解释

```python
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
gs = GridSearchCV(model, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
gs.fit(X_train, y_train)
print(gs.best_params_)
```

## 注意事项

1. GridSearchCV 在参数多时计算量指数增长
2. RandomizedSearchCV 更实用（控制搜索预算）
3. 调参必须用交叉验证，不能用测试集

## 总结与延伸

以上代码展示了 **超参数调优** 的完整流程。

**解读要点**：
- 关注 **ROC 曲线** 下的面积（AUC）
- 观察 **交叉验证** 各折的方差
- 对比不同评估指标的适用场景

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **贝叶斯优化**：Optuna、Hyperopt
- **早停调参**：连续 N 轮不提升就停止
- **自动机器学习（AutoML）**：自动选择模型和超参数
